## Introduction

In this tutorial, we demonstrate the training procedure of TIGON on EMT dataset. The EMT dataset is time-series scRNA-seq dataset from an A549 cancer cell line where cells were exposed to TGFB1 to induce EMT at the first five time points [1].

Here we have processed the dataset into 3-dimensional UMAP space, and use the UMAP embedding space as the input.

References:
1. Cook, D.P. and B.C. Vanderhyden, Context specificity of the EMT transcriptional response. Nature communications, 2020. 11(1): p. 2142

In [11]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA


import sys
sys.path.append('../')
from utility import *
#from selex import *

Here we use a training with 5 iterations as an example. 
GPU is suggested for training.

In [ ]:
args = create_args()

In [8]:
def Sampling_selex(num_samples,time_all,time_pt,data_train,counts,sigma,device):
    #perturb the  coordinate x with Gaussian noise N (0, sigma*I )
    mu = data_train[time_all[time_pt]]
    count=counts[time_all[time_pt]]
    count = np.log1p(count)
    count/=np.sum(count)
    num_gaussian = mu.shape[0] # mu is number_sample * dimension
    dim = mu.shape[1]
    sigma_matrix = sigma * torch.eye(dim)
    m = torch.distributions.multivariate_normal.MultivariateNormal(torch.zeros(dim), sigma_matrix)
    noise_add = m.rsample(torch.Size([num_samples])).type(torch.float32).to(device)
    # check if number of points is <num_samples
    count = torch.as_tensor(count, dtype=torch.float32)
    idx = torch.multinomial(count, num_samples, replacement=True)
    samples = mu[idx] + noise_add

    return samples


def MultimodalGaussian_density_selex(x,time_all,time_pt,data_train,counts,sigma,device):
    """density function for MultimodalGaussian
    """
    mu = data_train[time_all[time_pt]]
    count=counts[time_all[time_pt]]
    count = np.log1p(count)
    num_gaussian = mu.shape[0] # mu is number_sample * dimension
    dim = mu.shape[1]
    sigma_matrix = sigma * torch.eye(dim).type(torch.float32).to(device)
    p_unn = torch.zeros([x.shape[0]]).type(torch.float32).to(device)
    for i in range(num_gaussian):
        m = torch.distributions.multivariate_normal.MultivariateNormal(mu[i,:], sigma_matrix)
        p_unn = p_unn + count[i]*torch.exp(m.log_prob(x)).type(torch.float32).to(device)
    p_n = p_unn/num_gaussian
    return p_n

def knn_to_grid(df,survivor,X,k):
    df=df[df['Sequence'].isin(survivor)]
    survivor_index=df.index.to_list()
    X = np.ascontiguousarray(X.astype(np.float32)) 
    d = X.shape[1] 
    index = faiss.IndexFlatL2(d) 
    index.add(X) 
    query_subset = X[survivor_index]
    distances, indices = index.search(query_subset, k)
    sub=X[indices]
    grid=np.mean(sub,axis=1)
    return distances,indices,np.asanyarray(grid)

def all_t_process(df,X,count,survivors,time,n_componets,k):#(time,name,dir,k,device,n_componets):
    #survivors=pd.read_csv(f'{dir}/survivors.csv')
    survivors = survivors.rename(columns={"sequence": "Sequence"})
    survivors = survivors.rename(columns={"count": "Count"})
    survivors=survivors['Sequence'].to_list()
    data=[]
    extra=[]
    pca = PCA(n_components=n_componets)
    for t in time:
        #df=pd.read_csv(f'{dir}/{name}{(t):02d}_2d/cleaned.csv')
        #X=np.load(f'{dir}/{name}{(t):02d}_2d/descriptor.npz')['descriptor']
        #count=np.load(f'{dir}/{name}{(t):02d}_2d/descriptor.npz')['counts']
        X_reduced = pca.fit_transform(X)
        dist,ind,grid=knn_to_grid(df,survivors,X_reduced,k)
        data.append(torch.from_numpy(grid).type(torch.float32).to(device))
        mean=np.mean(count[ind],axis=1)
        extra.append(mean/np.sum(mean))
    return data,extra




In [ ]:
df=pd.read_csv(f'../doxycol/doxycol10_2d/cleaned.csv')[:10000]
X=np.load(f'../doxycol/doxycol10_2d/descriptor.npz')['descriptor'][:10000]
count=np.load(f'../doxycol/doxycol10_2d/descriptor.npz')['counts'][:10000]

In [10]:
survivors=pd.read_csv(f'../doxycol//survivors.csv')

In [12]:
if __name__ == '__main__':
    
    random.seed(args.seed)
    torch.manual_seed(args.seed)

    device = torch.device('cuda:' + str(args.gpu)
                            if torch.cuda.is_available() else 'cpu')
    # load dataset
    integral_time = args.timepoints
    data_train,count_train=all_t_process(df,X,count,survivors,integral_time,args.n_componets,args.k)#(integral_time,args.dataset,args.input_dir,args.k,device,args.n_componets)

    time_pts = range(len(data_train))
    leave_1_out = []
    train_time = [x for i,x in enumerate(time_pts) if i!=leave_1_out]


    # model
    func = UOT(in_out_dim=data_train[0].shape[1], hidden_dim=args.hidden_dim,n_hiddens=args.n_hiddens,activation=args.activation).to(device)
    func.apply(initialize_weights)


    # configure training options
    options = {}
    options.update({'method': 'Dopri5'})
    options.update({'h': None})
    options.update({'rtol': 1e-3})
    options.update({'atol': 1e-5})
    options.update({'print_neval': False})
    options.update({'neval_max': 1000000})
    options.update({'safety': None})

    optimizer = optim.Adam(func.parameters(), lr=args.lr, weight_decay= 0.01)
    lr_adjust = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[args.niters-400,args.niters-200], gamma=0.5, last_epoch=-1)
    mse = nn.MSELoss()

    LOSS = []
    L2_1 = []
    L2_2 = []
    Trans = []
    Sigma = []
    
    if args.save_dir is not None:
        if not os.path.exists(args.save_dir):
            os.makedirs(args.save_dir)
        ckpt_path = os.path.join(args.save_dir, 'ckpt.pth')
        if os.path.exists(ckpt_path):
            checkpoint = torch.load(ckpt_path)
            func.load_state_dict(checkpoint['func_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            print('Loaded ckpt from {}'.format(ckpt_path))

    try:
        sigma_now = 1
        for itr in range(1, args.niters + 1):
            optimizer.zero_grad()
            
            loss, loss1, sigma_now, L2_value1, L2_value2 = train_model(mse,func,args,data_train,count_train,train_time,integral_time,sigma_now,options,device,itr)

            
            loss.backward()
            optimizer.step()
            lr_adjust.step()

            LOSS.append(loss.item())
            Trans.append(loss1[-1].mean(0).item())
            Sigma.append(sigma_now)
            L2_1.append(L2_value1.tolist())
            L2_2.append(L2_value2.tolist())
            
            print('Iter: {}, loss: {:.4f}'.format(itr, loss.item()))
            
            
            if itr % 500 == 0:
                ckpt_path = os.path.join(args.save_dir, 'ckpt_itr{}.pth'.format(itr))
                torch.save({'func_state_dict': func.state_dict()}, ckpt_path)
                print('Iter {}, Stored ckpt at {}'.format(itr, ckpt_path))
                
            
            

    except KeyboardInterrupt:
        if args.save_dir is not None:
            ckpt_path = os.path.join(args.save_dir, 'ckpt.pth')
            torch.save({
                'func_state_dict': func.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, ckpt_path)
            print('Stored ckpt at {}'.format(ckpt_path))
    print('Training complete after {} iters.'.format(itr))
    
    
    ckpt_path = os.path.join(args.save_dir, 'ckpt.pth')
    torch.save({
        'func_state_dict': func.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'LOSS':LOSS,
        'TRANS':Trans,
        'L2_1': L2_1,
        'L2_2': L2_2,
        'Sigma': Sigma
    }, ckpt_path)
    print('Stored ckpt at {}'.format(ckpt_path))

Iter: 1, loss: 171.3319
Iter: 2, loss: 48.0599
Iter: 3, loss: 17.3190
Iter: 4, loss: 7.9062
Iter: 5, loss: 4.4989
Training complete after 5 iters.
Stored ckpt at ../doxy_out/ckpt.pth
